In [ ]:
def sort_by_taxonomy(uniprot_data: Dict[str, Dict]) -> List[Dict]:
    """Sort the UniProt data by taxonomy ID and organism name"""
    sorted_data = sorted(
        uniprot_data.items(),
        key=lambda x: (x[1]['taxonomy_id'], x[1]['organism'])
    )
    return [{'id': k, **v} for k, v in sorted_data]

def get_taxonomy_lineage(taxid: int) -> Dict:
    """Get full taxonomic lineage from NCBI Taxonomy using E-utilities"""
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    efetch_url = f"{base_url}efetch.fcgi?db=taxonomy&id={taxid}&format=xml"
    
    try:
        response = requests.get(efetch_url)
        if response.status_code == 200:
            root = ET.fromstring(response.text)
            lineage_elem = root.find('.//Lineage')
            if lineage_elem is not None:
                lineage = lineage_elem.text.split('; ')
                if lineage[0] == "cellular organisms":
                    lineage = lineage[1:]
                formatted_lineage = '/'.join(lineage)
                return {
                    'taxid': taxid,
                    'lineage': formatted_lineage
                }
    except Exception as e:
        print(f"Error fetching taxonomy for {taxid}: {e}")
    
    return {'taxid': taxid, 'lineage': ''}

In [1]:
import random

# Generate random taxids (e.g., 9606 for human, 10090 for mouse, etc.)
random_taxids = [str(random.randint(1000, 999999)) for _ in range(5)]

# Create a simple Newick tree string
# Example: (9606,10090,10116,7227,7955);
tree_str = f"({','.join(random_taxids)});"

# Save to file
with open("random_taxid_tree.nwk", "w") as f:
    f.write(tree_str)

In [4]:
from itolapi import Itol
from pathlib import Path

itol_uploader = Itol()
itol_uploader.add_file(Path('random_taxid_tree.nwk'))
itol_uploader.params['treeName'] = 'Random TaxID Tree'
itol_uploader.params['APIkey'] = '1grVqxyHGGLxpNnbAOdbAw'  
status = itol_uploader.upload()
if status:
    print("Upload successful!")
    print("Tree ID:", itol_uploader.comm.tree_id)
    print("Webpage:", itol_uploader.get_webpage())
else:
    print("Upload failed.")
    print(itol_uploader.comm.upload_output)

Upload failed.
ERR 0: No valid subscription detected for API key 1grVqxyHGGLxpNnbAOdbAw. Please check https://itol.embl.de/infoReg.cgi
